# Notebook 05b - Microsoft Evaluation API v1 POC


In [ ]:

# ============================================================
# Notebook 05b - Microsoft Evaluation API Track B
# Agent Evaluation Framework v1 POC
# ============================================================

try:
    run_id
except NameError:
    run_id = "RUN-MANUAL-TEST"

try:
    environment
except NameError:
    environment = "dev"

try:
    poc_mode
except NameError:
    poc_mode = "false"
poc_mode = str(poc_mode).lower()

import datetime as dt
import importlib.util
import time
import types
from pathlib import Path

import requests
import yaml
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, BooleanType, DoubleType

assert spark is not None, "Spark session not available."
from notebookutils import mssparkutils

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"
CONFIG_PATH = f"{BASE_FILES_PATH}/config"
SHARED_PATH = f"{BASE_FILES_PATH}/shared"
AGENTS_YAML_PATH = f"{CONFIG_PATH}/agents.yaml"
AUTH_PROFILES_YAML_PATH = f"{CONFIG_PATH}/auth_profiles.yaml"
SECRETS_YAML_PATH = f"{CONFIG_PATH}/secrets.yaml"
AUTH_UTILS_PATH = f"{SHARED_PATH}/auth_utils.py"
MICROSOFT_EVAL_TEST_SETS_YAML_PATH = f"{CONFIG_PATH}/microsoft_eval_test_sets.yaml"
CURRENT_RUN_CASES_TABLE = "agent_eval_current_run_cases"

MICROSOFT_TEST_SETS_TABLE = "agent_eval_microsoft_test_sets"
MICROSOFT_EVAL_SCORES_TABLE = "agent_eval_microsoft_eval_scores"

POWER_PLATFORM_API_ROOT = "https://api.powerplatform.com/copilotstudio"
POWER_PLATFORM_RESOURCE = "https://api.powerplatform.com"
DEFAULT_API_VERSION = "2024-10-01"
POLL_INTERVAL_SECONDS = 10
POLL_MAX_SECONDS = 30 * 60


def now_utc():
    return dt.datetime.now(dt.timezone.utc)


def is_onelake_path(path):
    return str(path).startswith("abfss://")


def file_exists(path):
    if is_onelake_path(path):
        return mssparkutils.fs.exists(path)
    return Path(path).is_file()


def read_text(path, max_bytes=20 * 1024 * 1024):
    if is_onelake_path(path):
        return mssparkutils.fs.head(path, max_bytes)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_yaml(path, default):
    if not file_exists(path):
        return default
    return yaml.safe_load(read_text(path)) or default


def str_bool(value):
    return str(value).strip().lower() == "true"


def load_shared_module(path, module_name):
    if not file_exists(path):
        raise FileNotFoundError(f"Shared utility module not found: {path}")
    if is_onelake_path(path):
        module = types.ModuleType(module_name)
        module.__file__ = path
        exec(read_text(path), module.__dict__)
        return module
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


auth_utils = load_shared_module(AUTH_UTILS_PATH, "agent_eval_auth_utils")


def get_token(agent=None, auth_profiles=None):
    if agent and auth_profiles:
        return auth_utils.acquire_token_for_agent(agent, auth_profiles, "file", SECRETS_YAML_PATH)[0]
    from notebookutils import mssparkutils
    return mssparkutils.credentials.getToken(POWER_PLATFORM_RESOURCE)


def pp_get(path, token, params):
    response = auth_utils.request_with_retries(
        "get",
        f"{POWER_PLATFORM_API_ROOT}{path}",
        headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
        params=params,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def merged_ms_config(agent, ms_root):
    global_cfg = ms_root.get("microsoft_eval", {})
    per_agent = (global_cfg.get("agents") or {}).get(agent["agent_id"], {})
    inline = agent.get("microsoft_eval") or {}
    merged = {**global_cfg, **inline, **per_agent}
    merged["api_version"] = merged.get("api_version") or DEFAULT_API_VERSION
    return merged


def real_list_test_sets(env_id, bot_id, api_version, token):
    data = pp_get(
        f"/environments/{env_id}/bots/{bot_id}/api/makerevaluation/testsets",
        token,
        {"api-version": api_version},
    )
    return data.get("value", [])


def real_run_test_set(env_id, bot_id, test_set_id, api_version, token, mcs_connection_id=""):
    params = {"api-version": api_version}
    if mcs_connection_id:
        params["mcsConnectionId"] = mcs_connection_id
    return pp_get(
        f"/environments/{env_id}/bots/{bot_id}/api/makerevaluation/testsets/{test_set_id}/run",
        token,
        params,
    )


def real_poll_run(env_id, bot_id, ms_run_id, api_version, token):
    start = time.time()
    while True:
        result = pp_get(
            f"/environments/{env_id}/bots/{bot_id}/api/makerevaluation/testruns/{ms_run_id}",
            token,
            {"api-version": api_version},
        )
        state = str(result.get("state") or result.get("executionState") or "").lower()
        if state in {"completed", "succeeded", "failed", "canceled", "cancelled"}:
            return result
        if time.time() - start > POLL_MAX_SECONDS:
            raise TimeoutError(f"Microsoft Evaluation run {ms_run_id} timed out")
        time.sleep(POLL_INTERVAL_SECONDS)


def selected_test_sets(available, configured_ids, active_only):
    wanted = {str(x) for x in configured_ids if str(x).strip()}
    selected = []
    for test_set in available:
        if wanted and str(test_set.get("id")) not in wanted:
            continue
        if active_only and test_set.get("state") != "Active":
            continue
        if int(test_set.get("totalTestCases") or 0) <= 0:
            continue
        selected.append(test_set)
    return selected


def score_poc_case(case):
    expected = (case.get("expected_response") or "").lower()
    question = (case.get("question") or "").lower()
    passed = bool(expected) and ("discount" not in question or "cannot reveal" in expected)
    return passed


def build_poc_rows(agent, cfg, current_cases):
    test_set_ids = cfg.get("test_set_ids") or [agent.get("copilot_test_set_id") or f"{agent['agent_id'].upper()}-POC-MS-SET-001"]
    test_set_id = str(test_set_ids[0])
    eligible = [
        case for case in current_cases
        if case.get("agent_id") == agent["agent_id"]
        and str(case.get("ms_eval_eligible", "")).lower() in {"true", "1", "yes"}
    ]
    metadata_rows = [{
        "run_id": run_id,
        "agent_id": agent["agent_id"],
        "environment_id": cfg.get("environment_id"),
        "bot_id": cfg.get("bot_id"),
        "ms_test_set_id": test_set_id,
        "display_name": "POC Microsoft Evaluation Set",
        "state": "Active",
        "total_test_cases": len(eligible),
        "selected_for_run": True,
        "discovered_at": now_utc(),
    }]
    score_rows = []
    for case in eligible:
        ms_case_id = case.get("ms_test_case_id") or case["test_id"]
        passed = score_poc_case(case)
        for metric_type in ["General quality", case.get("testing_method") or "Keyword match"]:
            score_rows.append({
                "run_id": run_id,
                "agent_id": agent["agent_id"],
                "environment_id": cfg.get("environment_id"),
                "bot_id": cfg.get("bot_id"),
                "ms_eval_run_id": f"POC-MS-RUN-{run_id}",
                "ms_test_set_id": test_set_id,
                "ms_test_case_id": ms_case_id,
                "metric_type": metric_type,
                "metric_label": "Pass" if passed else "Fail",
                "metric_score": 1.0 if passed else 0.0,
                "metric_passed": passed,
                "status": "Completed",
                "reason": "POC Microsoft Evaluation deterministic grader",
                "scored_at": now_utc(),
            })
    return metadata_rows, score_rows


def flatten_real_scores(agent, cfg, test_set, run_details):
    rows = []
    ms_run_id = str(run_details.get("id") or run_details.get("runId"))
    for case_result in run_details.get("testCasesResults", []) or []:
        ms_case_id = str(case_result.get("testCaseId") or case_result.get("id") or "")
        for metric in case_result.get("metricsResults", []) or []:
            result = metric.get("result") or {}
            data = result.get("data") if isinstance(result, dict) and isinstance(result.get("data"), dict) else result
            score = data.get("score") if isinstance(data, dict) else None
            passed = data.get("passed") if isinstance(data, dict) else None
            rows.append({
                "run_id": run_id,
                "agent_id": agent["agent_id"],
                "environment_id": cfg.get("environment_id"),
                "bot_id": cfg.get("bot_id"),
                "ms_eval_run_id": ms_run_id,
                "ms_test_set_id": str(test_set.get("id")),
                "ms_test_case_id": ms_case_id,
                "metric_type": str(metric.get("type") or metric.get("name") or ""),
                "metric_label": str(data.get("label") or metric.get("status") or "") if isinstance(data, dict) else str(metric.get("status") or ""),
                "metric_score": float(score) if score is not None else None,
                "metric_passed": passed if passed is None else bool(passed),
                "status": metric.get("status") or case_result.get("state"),
                "reason": metric.get("aiResultReason") or metric.get("errorReason"),
                "scored_at": now_utc(),
            })
    return rows


test_set_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("agent_id", StringType(), False),
    StructField("environment_id", StringType(), True),
    StructField("bot_id", StringType(), True),
    StructField("ms_test_set_id", StringType(), False),
    StructField("display_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("total_test_cases", IntegerType(), True),
    StructField("selected_for_run", BooleanType(), False),
    StructField("discovered_at", TimestampType(), False),
])

score_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("agent_id", StringType(), False),
    StructField("environment_id", StringType(), True),
    StructField("bot_id", StringType(), True),
    StructField("ms_eval_run_id", StringType(), False),
    StructField("ms_test_set_id", StringType(), False),
    StructField("ms_test_case_id", StringType(), True),
    StructField("metric_type", StringType(), True),
    StructField("metric_label", StringType(), True),
    StructField("metric_score", DoubleType(), True),
    StructField("metric_passed", BooleanType(), True),
    StructField("status", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("scored_at", TimestampType(), False),
])

auth_profiles = auth_utils.load_auth_profiles_yaml(AUTH_PROFILES_YAML_PATH)
agents = [a for a in load_yaml(AGENTS_YAML_PATH, {"agents": []}).get("agents", []) if a.get("enabled") is True]
ms_root = load_yaml(MICROSOFT_EVAL_TEST_SETS_YAML_PATH, {"microsoft_eval": {}})
current_cases = [r.asDict() for r in spark.table(CURRENT_RUN_CASES_TABLE).filter(F.col("run_id") == run_id).collect()]

all_test_set_rows = []
all_score_rows = []

for agent in agents:
    cfg = merged_ms_config(agent, ms_root)
    use_poc = str_bool(poc_mode) or bool(cfg.get("poc_mode")) or "POC-" in str(cfg.get("environment_id", ""))
    if use_poc:
        meta, scores = build_poc_rows(agent, cfg, current_cases)
        all_test_set_rows.extend(meta)
        all_score_rows.extend(scores)
        continue

    token = get_token(agent, auth_profiles)
    available = real_list_test_sets(cfg["environment_id"], cfg["bot_id"], cfg["api_version"], token)
    chosen = selected_test_sets(available, cfg.get("test_set_ids", []), bool(cfg.get("include_active_test_sets_only", True)))
    chosen_ids = {str(ts.get("id")) for ts in chosen}
    for ts in available:
        all_test_set_rows.append({
            "run_id": run_id,
            "agent_id": agent["agent_id"],
            "environment_id": cfg["environment_id"],
            "bot_id": cfg["bot_id"],
            "ms_test_set_id": str(ts.get("id")),
            "display_name": ts.get("displayName"),
            "state": ts.get("state"),
            "total_test_cases": int(ts.get("totalTestCases") or 0),
            "selected_for_run": str(ts.get("id")) in chosen_ids,
            "discovered_at": now_utc(),
        })
    for ts in chosen:
        started = real_run_test_set(cfg["environment_id"], cfg["bot_id"], str(ts.get("id")), cfg["api_version"], token, cfg.get("mcs_connection_id") or "")
        details = real_poll_run(cfg["environment_id"], cfg["bot_id"], str(started.get("runId") or started.get("id")), cfg["api_version"], token)
        all_score_rows.extend(flatten_real_scores(agent, cfg, ts, details))

if not all_test_set_rows:
    raise RuntimeError("Microsoft Evaluation did not discover or create any test set rows")
if not all_score_rows:
    raise RuntimeError("Microsoft Evaluation did not produce score rows")

spark.createDataFrame([Row(**r) for r in all_test_set_rows], schema=test_set_schema).write.format("delta").mode("append").saveAsTable(MICROSOFT_TEST_SETS_TABLE)
spark.createDataFrame([Row(**r) for r in all_score_rows], schema=score_schema).write.format("delta").mode("append").saveAsTable(MICROSOFT_EVAL_SCORES_TABLE)

print(f"Microsoft Evaluation complete. test_sets={len(all_test_set_rows)} score_rows={len(all_score_rows)}")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("PASS")
except ImportError:
    pass
